<a href="https://colab.research.google.com/github/Hesam-s/NLP/blob/main/NPC_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0: Importing Libraries

In [ ]:
import kagglehub
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import gc
import torch
import torch.nn as nn
from transformers import DistilBertTokenizerFast, DistilBertModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.distributions import Categorical

print("All libraries are ready to use!")

All libraries are ready to use!


# Step 1: Data Collection

In [ ]:
# --- 1. CONFIGURATION ---
# Using the correct public handle for this dataset
dataset_handle = "zara2099/context-aware-npc-behavior-dataset"
target_csv_name = "npc_behavior_dataset.csv"

# --- 2. DOWNLOAD ---
try:
    download_path = kagglehub.dataset_download(dataset_handle)
    print(f"📍 Original download path: {download_path}")

    # --- 3. MOVE & CLEANUP ---
    # Find the csv in the downloaded folder (it might have a different name in the zip)
    files = os.listdir(download_path)
    source_file = None
    for f in files:
        if f.endswith('.csv'):
            source_file = os.path.join(download_path, f)
            break

    local_destination = os.path.join("/content", target_csv_name)

    if source_file and os.path.exists(source_file):
        shutil.copy(source_file, local_destination)
        print(f"✅ Successfully moved data to: {local_destination}")
    else:
        print(f"❌ Error: CSV file not found in {download_path}")

except Exception as e:
    print(f"❌ Download failed: {e}")
    print("Tip: If this is a private dataset, you must set KAGGLE_USERNAME and KAGGLE_KEY in your environment secrets.")

Using Colab cache for faster access to the 'context-aware-npc-behavior-dataset' dataset.
📍 Original download path: /kaggle/input/context-aware-npc-behavior-dataset
✅ Successfully moved data to: /content/npc_behavior_dataset.csv


# Step 2: Data Preproccessing

In [ ]:
# --- 1. LOAD DATA ---
df = pd.read_csv('npc_behavior_dataset.csv')

# --- 2. CLEAN CATEGORICAL TEXT ---
cat_cols = ['Environment_State', 'Opponent_Strategy', 'NPC_Action_Type']
label_encoders = {}

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    df[f'{col}_ID'] = le.fit_transform(df[col])
    label_encoders[col] = le

# --- 3. SCALE NUMERIC COLUMNS (FOR RL ONLY) ---
num_cols = [
    'Sensory_Input_Level',
    'Decision_Time',
    'Policy_Confidence',
    'Reward_Score',
    'Human_Likeness_Score',
    'Behavioral_Diversity'
]

scaler = MinMaxScaler()
scaled_cols = [f'Scaled_{c}' for c in num_cols]
df[scaled_cols] = scaler.fit_transform(df[num_cols])

# --- 4. BUCKETIZATION (LANGUAGE-FRIENDLY) ---
def bucket(v):
    if v < 0.33:
        return "low"
    elif v < 0.66:
        return "medium"
    else:
        return "high"

df['sensory_level'] = df['Scaled_Sensory_Input_Level'].apply(bucket)
df['decision_speed'] = df['Scaled_Decision_Time'].apply(
    lambda x: "fast" if x < 0.33 else "moderate" if x < 0.66 else "slow"
)
df['confidence_level'] = df['Scaled_Policy_Confidence'].apply(bucket)
df['diversity_level'] = df['Scaled_Behavioral_Diversity'].apply(bucket)

# --- 5. BERT TEXTUALIZATION (SEMANTIC, NOT NUMERIC) ---
df['bert_input_state'] = df.apply(lambda x: (
    f"Environment: {x['Environment_State']}. "
    f"Opponent strategy: {x['Opponent_Strategy']}. "
    f"Sensory awareness is {x['sensory_level']}. "
    f"Decision making is {x['decision_speed']}. "
    f"Policy confidence is {x['confidence_level']}. "
    f"Behavioral diversity is {x['diversity_level']}."
), axis=1)

# --- 6. FINAL SANITY CHECK ---
print("✅ Preprocessing complete\n")

print("--- Action Label Mapping ---")
print(dict(zip(
    label_encoders['NPC_Action_Type'].classes_,
    label_encoders['NPC_Action_Type'].transform(
        label_encoders['NPC_Action_Type'].classes_
    )
)))

# --- 7. DISPLAY MASTER VIEW ---
cols_to_display = [
    'bert_input_state',
    'NPC_Action_Type_ID',
    'Scaled_Reward_Score',
    'Scaled_Human_Likeness_Score'
]

df[cols_to_display].head(20)


✅ Preprocessing complete

--- Action Label Mapping ---
{'attack': np.int64(0), 'communicate': np.int64(1), 'defend': np.int64(2), 'explore': np.int64(3), 'hide': np.int64(4)}


,bert_input_state,NPC_Action_Type_ID,Scaled_Reward_Score,Scaled_Human_Likeness_Score
0,Environment: stealth. Opponent strategy: defen...,4,0.595960,0.837474
1,Environment: combat. Opponent strategy: balanc...,3,0.989899,0.322613
2,Environment: calm. Opponent strategy: aggressi...,1,0.616162,0.889077
3,Environment: stealth. Opponent strategy: aggre...,0,0.292929,0.160131
4,Environment: stealth. Opponent strategy: aggre...,1,0.929293,0.172531
5,Environment: combat. Opponent strategy: aggres...,0,0.212121,0.266352
6,Environment: calm. Opponent strategy: random. ...,2,0.747475,0.118996
7,Environment: calm. Opponent strategy: aggressi...,0,0.838384,0.229871
8,Environment: stealth. Opponent strategy: rando...,4,0.282828,0.105867
9,Environment: chaotic. Opponent strategy: rando...,3,0.404040,0.821764


# Step 3: Creating a Master Stack

In [ ]:
# --- 1. THE MASTER STACK ---
# X_raw: All environmental and numerical context combined into sentences for BERT
X_raw = df['bert_input_state'].values

# y_labels: The categorical Action IDs (0-4) for the Actor Head
y_labels = df['NPC_Action_Type_ID'].values

# RL Metrics: Scaled scores for the Reward and Human-Likeness alignment
rewards = df['Scaled_Reward_Score'].values
likeness = df['Scaled_Human_Likeness_Score'].values

# --- 2. VERIFY THE STACK ---
print(f"✅ Master Stack Created Successfully!")
print(f"Total X (States) Samples: {X_raw.shape[0]}")
print(f"Total y (Actions) Samples: {y_labels.shape[0]}")

# Display the first 20 rows of the Master Stack
pd.DataFrame({
    'X_raw (Input State)': X_raw,
    'y_labels (Action ID)': y_labels,
    'Reward': rewards,
    'Likeness': likeness
}).head(20)

✅ Master Stack Created Successfully!
Total X (States) Samples: 1000
Total y (Actions) Samples: 1000


,X_raw (Input State),y_labels (Action ID),Reward,Likeness
0,Environment: stealth. Opponent strategy: defen...,4,0.595960,0.837474
1,Environment: combat. Opponent strategy: balanc...,3,0.989899,0.322613
2,Environment: calm. Opponent strategy: aggressi...,1,0.616162,0.889077
3,Environment: stealth. Opponent strategy: aggre...,0,0.292929,0.160131
4,Environment: stealth. Opponent strategy: aggre...,1,0.929293,0.172531
5,Environment: combat. Opponent strategy: aggres...,0,0.212121,0.266352
6,Environment: calm. Opponent strategy: random. ...,2,0.747475,0.118996
7,Environment: calm. Opponent strategy: aggressi...,0,0.838384,0.229871
8,Environment: stealth. Opponent strategy: rando...,4,0.282828,0.105867
9,Environment: chaotic. Opponent strategy: rando...,3,0.404040,0.821764


# Step 4: Data Splitting

In [ ]:
# --- Data SPLIT ---
# We split X, y, rewards, and likeness simultaneously to maintain perfect index alignment
X_train, X_test, y_train, y_test, r_train, r_test, l_train, l_test = train_test_split(
    X_raw,
    y_labels,
    rewards,
    likeness,
    test_size=0.2,
    random_state=42
)

# --- VERIFICATION ---
print(f"✅ Split Complete!")
print("-" * 30)
print(f"Training Data: {len(X_train)} rows (Used for BERT/PPO/GRPO Training)")
print(f"Testing Data:  {len(X_test)} rows (Used for evaluation)")
print("-" * 30)

# Example of a single training record
print("\n[Sample Record Index 0]")
print(f"Input text: {X_train[0]}")
print(f"Target Action ID: {y_train[0]}")
print(f"Expected Reward: {r_train[0]:.4f}")

✅ Split Complete!
------------------------------
Training Data: 800 rows (Used for BERT/PPO/GRPO Training)
Testing Data:  200 rows (Used for evaluation)
------------------------------

[Sample Record Index 0]
Input text: Environment: combat. Opponent strategy: aggressive. Sensory awareness is medium. Decision making is slow. Policy confidence is high. Behavioral diversity is medium.
Target Action ID: 3
Expected Reward: 0.9798


# Step 5: DistilBERT Model Implementation

In [ ]:
# ======================
# 1. NPC MIND
# ======================
class NPCMind(nn.Module):
    def __init__(self, num_actions=5, dropout_rate=0.2):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # UNFREEZE EVERYTHING
        for param in self.encoder.parameters():
            param.requires_grad = True

        hidden_size = self.encoder.config.hidden_size  # 768

        self.dropout = nn.Dropout(dropout_rate)

        self.actor = nn.Linear(hidden_size, num_actions)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids,
                               attention_mask=attention_mask)

        # CLS token
        cls = outputs.last_hidden_state[:, 0]
        cls = self.dropout(cls)

        logits = self.actor(cls)
        value = self.critic(cls).squeeze(-1)

        return logits, value

# ======================
# 2. DATA PREPARATION
# ======================
class NPCDataset(Dataset):
    def __init__(self, texts, labels, rewards, likeness, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.rewards = list(rewards)
        self.likeness = list(likeness)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "rewards": torch.tensor(self.rewards[idx], dtype=torch.float),
            "likeness": torch.tensor(self.likeness[idx], dtype=torch.float),
        }

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Distribution check
dist = np.bincount(y_train) / len(y_train)
print(f"✅ Action distribution: {dist}")

train_dataset = NPCDataset(X_train, y_train, r_train, l_train, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

✅ Action distribution: [0.1925  0.215   0.20375 0.175   0.21375]


# Step 6: Model Creation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NPCMind(num_actions=5).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ DistilBERT ready on {device}")
print(f"Trainable Parameters: {trainable_params:,}")
print("Strategy: Full fine-tuning (lr=2e-5)")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ DistilBERT ready on cuda
Trainable Parameters: 66,367,494
Strategy: Full fine-tuning (lr=2e-5)


# Step 7: Supervised Fine-Tuning (SFT)

In [ ]:
epochs = 10
ce_loss_fn = nn.CrossEntropyLoss()

print(f"🚀 Pure SFT Grounding on {device}...")

for epoch in range(epochs):
    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        logits, values = model(input_ids, attention_mask)

        loss = ce_loss_fn(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc = total_correct / total_samples

    print(f"Epoch {epoch+1:02d} | CE: {avg_loss:.4f} | Acc: {acc*100:.2f}%")

🚀 Pure SFT Grounding on cuda...
Epoch 01 | CE: 1.6432 | Acc: 17.75%
Epoch 02 | CE: 1.6208 | Acc: 18.88%
Epoch 03 | CE: 1.6230 | Acc: 21.38%
Epoch 04 | CE: 1.6189 | Acc: 21.12%
Epoch 05 | CE: 1.6078 | Acc: 20.38%
Epoch 06 | CE: 1.6035 | Acc: 23.75%
Epoch 07 | CE: 1.6094 | Acc: 25.25%
Epoch 08 | CE: 1.6043 | Acc: 22.62%
Epoch 09 | CE: 1.6018 | Acc: 24.50%
Epoch 10 | CE: 1.5854 | Acc: 25.87%


In [ ]:
# --- 4. RAM CLEANING ---
vars_to_kill = ['df', 'X_train', 'X_test', 'y_train', 'y_test', 'r_train', 'r_test', 'l_train', 'l_test']
for var in vars_to_kill:
    if var in globals(): # Use globals() to be safe in notebook cells
        del globals()[var]

gc.collect()
print("🧹 RAM Cleaned! Ready for training.")

# Step 8: Testing Before PPO and GRPO
## Step 8.1: Greedy + Probabilistic Decision Test

In [ ]:
model.eval()

def test_policy_decision(text, tokenizer, model, device):
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        logits = model(
            encoding["input_ids"],
            encoding["attention_mask"]
        )
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    action_id = int(np.argmax(probs))
    return action_id, probs


# 🔎 TEST ON RANDOM TEST SAMPLES
print("🔍 Policy Decision Test (Test Set Samples)\n")

for i in range(5):
    text = test_dataset.texts[i]
    true_action = test_dataset.labels[i]

    action, probs = test_policy_decision(text, tokenizer, model, device)

    print(f"State: {text}")
    print(f"True Action: {true_action}")
    print(f"Predicted Action: {action}")
    print(f"Action Probabilities: {np.round(probs, 3)}")
    print("-" * 80)

🔍 Policy Decision Test (Test Set Samples)

State: Environment: stealth. Opponent strategy: aggressive. Sensory awareness is low. Decision making is fast. Policy confidence is low. Behavioral diversity is medium.
True Action: 2
Predicted Action: 0
Action Probabilities: [0.235 0.228 0.205 0.149 0.183]
--------------------------------------------------------------------------------
State: Environment: calm. Opponent strategy: balanced. Sensory awareness is medium. Decision making is moderate. Policy confidence is low. Behavioral diversity is medium.
True Action: 2
Predicted Action: 0
Action Probabilities: [0.235 0.228 0.205 0.149 0.183]
--------------------------------------------------------------------------------
State: Environment: stealth. Opponent strategy: random. Sensory awareness is high. Decision making is fast. Policy confidence is medium. Behavioral diversity is medium.
True Action: 1
Predicted Action: 0
Action Probabilities: [0.235 0.228 0.205 0.149 0.183]
-------------------

## Step 8.2: Sensitivity Test (does the model understand meaning?)

In [ ]:
base_state = (
    "Environment: forest. "
    "Opponent strategy: aggressive. "
    "Sensory awareness is medium. "
    "Decision making is moderate. "
    "Policy confidence is low. "
    "Behavioral diversity is medium."
)

variants = [
    base_state.replace("low", "high"),
    base_state.replace("aggressive", "defensive"),
    base_state.replace("moderate", "fast"),
]

print("🧠 Sensitivity Test\n")

for text in [base_state] + variants:
    action, probs = test_policy_decision(text, tokenizer, model, device)
    print(f"Input: {text}")
    print(f"Predicted Action: {action}")
    print(f"Probs: {np.round(probs, 3)}")
    print("-" * 80)

🧠 Sensitivity Test

Input: Environment: forest. Opponent strategy: aggressive. Sensory awareness is medium. Decision making is moderate. Policy confidence is low. Behavioral diversity is medium.
Predicted Action: 0
Probs: [0.235 0.228 0.205 0.149 0.183]
--------------------------------------------------------------------------------
Input: Environment: forest. Opponent strategy: aggressive. Sensory awareness is medium. Decision making is moderate. Policy confidence is high. Behavioral diversity is medium.
Predicted Action: 0
Probs: [0.235 0.228 0.205 0.149 0.183]
--------------------------------------------------------------------------------
Input: Environment: forest. Opponent strategy: defensive. Sensory awareness is medium. Decision making is moderate. Policy confidence is low. Behavioral diversity is medium.
Predicted Action: 0
Probs: [0.235 0.228 0.205 0.149 0.183]
--------------------------------------------------------------------------------
Input: Environment: forest. Opponen

# Step 9: Human-Readable Policy Report

In [ ]:
action_decoder = label_encoders['NPC_Action_Type'].inverse_transform

def human_readable_policy(text):
    action, probs = test_policy_decision(text, tokenizer, model, device)

    ranked = np.argsort(probs)[::-1]

    print("🧾 HUMAN-READABLE POLICY REPORT")
    print("-" * 60)
    print(f"State Description:\n{text}\n")
    print("Model Preference Ranking:")

    for rank, idx in enumerate(ranked):
        print(
            f"{rank+1}. {action_decoder([idx])[0]} "
            f"(prob={probs[idx]:.3f})"
        )

    print(f"\n✅ Final Chosen Action: {action_decoder([action])[0]}")
    print("-" * 60)


# 🧪 Run on a custom state
human_readable_policy(
    "Environment: urban street. "
    "Opponent strategy: defensive. "
    "Sensory awareness is high. "
    "Decision making is fast. "
    "Policy confidence is high. "
    "Behavioral diversity is low."
)

🧾 HUMAN-READABLE POLICY REPORT
------------------------------------------------------------
State Description:
Environment: urban street. Opponent strategy: defensive. Sensory awareness is high. Decision making is fast. Policy confidence is high. Behavioral diversity is low.

Model Preference Ranking:
1. attack (prob=0.235)
2. communicate (prob=0.228)
3. defend (prob=0.205)
4. hide (prob=0.183)
5. explore (prob=0.149)

✅ Final Chosen Action: attack
------------------------------------------------------------


# Step 10: PPO Implementation

In [ ]:
# ======================
# POLICY GRADIENT STEP (CORRECT FOR YOUR SETUP)
# ======================
def ppo_step(
    model,
    optimizer,
    batch,
    clip_eps=0.2,
    value_coef=0.5,
    entropy_coef=0.01,
    device="cuda"
):
    model.train()

    # Move data to hardware
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    actions = batch["labels"].to(device)
    rewards = batch["rewards"].to(device)

    # 1. Normalize Rewards (Crucial for Critic stability)
    rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

    # 2. Forward Pass
    logits, values = model(input_ids, attention_mask)

    # Flatten values to [Batch Size] to match rewards
    values = values.view(-1)

    # 3. Policy Distribution
    dist = Categorical(logits=logits)
    log_probs = dist.log_prob(actions)
    entropy = dist.entropy().mean()

    # 4. Old Policy Baseline (Detached)
    old_log_probs = log_probs.detach()

    # 5. Advantage Calculation
    # How much better was the reward than what we expected?
    advantages = rewards - values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # 6. PPO Clipped Objective
    ratio = torch.exp(log_probs - old_log_probs)
    clipped_ratio = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)

    policy_loss = -torch.min(
        ratio * advantages,
        clipped_ratio * advantages
    ).mean()

    # 7. Value Loss (Using the now-defined F)
    value_loss = F.mse_loss(values, rewards)

    # 8. Total Loss
    loss = policy_loss + (value_coef * value_loss) - (entropy_coef * entropy)

    # 9. Optimization
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return {
        "loss": loss.item(),
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "entropy": entropy.item()
    }

# Step 11: PPO Training

In [ ]:
# Optional: Lower the learning rate for PPO stability
for param_group in optimizer.param_groups:
    param_group['lr'] = 1e-5

print(f"✅ Ready for PPO. Learning rate adjusted to: {optimizer.param_groups[0]['lr']}")

# ======================
# TRAINING LOOP
# ======================
ppo_epochs = 10
print("🚀 PPO REINFORCEMENT TRAINING STARTED")

for epoch in range(ppo_epochs):
    metrics_log = {"loss": 0, "policy": 0, "value": 0, "entropy": 0}

    for batch in train_loader:
        m = ppo_step(
            model=model,
            optimizer=optimizer,
            batch=batch,
            device=device
        )
        metrics_log["loss"] += m["loss"]
        metrics_log["policy"] += m["policy_loss"]
        metrics_log["value"] += m["value_loss"]
        metrics_log["entropy"] += m["entropy"]

    n = len(train_loader)
    print(f"PPO Epoch {epoch+1}/{ppo_epochs} | "
          f"Loss: {metrics_log['loss']/n:.4f} | "
          f"Value Err: {metrics_log['value']/n:.4f} | "
          f"Entropy: {metrics_log['entropy']/n:.4f}")

print("✅ PPO Training Complete! Your NPC is now strategically optimized.")

✅ Ready for PPO. Learning rate adjusted to: 1e-05
🚀 PPO REINFORCEMENT TRAINING STARTED


ValueError: too many values to unpack (expected 2)

# Step 12: PPO Testing

In [ ]:
# ======================
# PPO TEST (IMPROVED)
# ======================
print("\n🔍 PPO POLICY TEST")

model.eval()
with torch.no_grad():
    for i in range(5):
        sample = test_dataset[i]

        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)

        outputs = model(input_ids, attention_mask)
        logits = outputs[0]

        probs = torch.softmax(logits, dim=-1).squeeze(0)
        entropy = -(probs * torch.log(probs + 1e-8)).sum().item()

        # greedy action
        greedy_action = torch.argmax(probs).item()

        # stochastic action (better reflects PPO behavior)
        sampled_action = torch.distributions.Categorical(probs).sample().item()

        # top-k probabilities
        topk_probs, topk_actions = torch.topk(probs, k=min(5, probs.size(0)))

        print("State:")
        print(sample["text"])
        print(f"Greedy Action:   {greedy_action}")
        print(f"Sampled Action: {sampled_action}")
        print(f"Policy Entropy: {entropy:.4f}")
        print("Top Actions (action: prob):")

        for a, p in zip(topk_actions.tolist(), topk_probs.tolist()):
            print(f"  {a}: {p:.3f}")

        print("-" * 80)


# Step 13: GRPO Implementation

In [ ]:
# ======================
# GRPO STEP
# ======================
def grpo_step(
    model,
    optimizer,
    batch,
    temperature=1.0,
    device="cuda"
):
    model.train()

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    chosen = batch["chosen_action"].to(device)
    rejected = batch["rejected_action"].to(device)

    logits, _ = model(input_ids, attention_mask)
    log_probs = F.log_softmax(logits / temperature, dim=-1)

    chosen_logp = log_probs.gather(1, chosen.unsqueeze(1)).squeeze()
    rejected_logp = log_probs.gather(1, rejected.unsqueeze(1)).squeeze()

    loss = -torch.log(torch.sigmoid(chosen_logp - rejected_logp)).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()